In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Tensor-network error mitigation (TEM): функція Qiskit від Algorithmiq
> **Note:** Qiskit Functions — це експериментальна функція, доступна лише для користувачів IBM Quantum&reg; Premium Plan, Flex Plan та On-Prem (через IBM Quantum Platform API) Plan. Вони перебувають у статусі попереднього випуску і можуть змінюватися.


<details>
<summary><b>Версії пакетів</b></summary>

Код на цій сторінці розроблений із використанням наступних вимог.
Рекомендуємо використовувати ці або новіші версії.

```
qiskit[all]~=2.3.0
qiskit-ibm-catalog~=0.11.0
```
</details>
## Огляд
Метод Tensor-network Error Mitigation (TEM) від Algorithmiq — це гібридний
квантово-класичний алгоритм, розроблений для виконання пом'якшення шуму
повністю на класичному етапі постобробки. За допомогою TEM користувач може
обчислювати очікувані значення спостережуваних величин, пом'якшуючи неминучі
помилки, спричинені шумом на квантовому обладнанні, з підвищеною точністю та
ефективністю витрат, що робить його дуже привабливим варіантом як для
квантових дослідників, так і для практиків в індустрії.

Метод полягає у побудові тензорної мережі, що представляє обернений
глобальний канал шуму, який впливає на стан квантового процесора, а потім
у застосуванні цього відображення до інформаційно повних результатів
вимірювань, отриманих від зашумленого стану, для отримання незміщених
оцінок спостережуваних величин.

Перевагою TEM є те, що він використовує інформаційно повні вимірювання для
надання доступу до великого набору пом'якшених очікуваних значень
спостережуваних величин і має оптимальний вибірковий overhead на квантовому
обладнанні, як описано у Filippov et
al. (2023), [arXiv:2307.11740](https://arxiv.org/abs/2307.11740), та Filippov
et al. (2024), [arXiv:2403.13542](https://arxiv.org/abs/2403.13542).
Вибірковий overhead стосується кількості додаткових вимірювань, необхідних для
ефективного пом'якшення помилок — це критичний фактор у здійсненності
квантових обчислень. Тому TEM має потенціал для забезпечення квантової
переваги у складних сценаріях, таких як застосування в галузях квантового
хаосу, фізики багатьох тіл, динаміки Габбарда та симуляцій хімії малих
молекул.

Основні особливості та переваги TEM можна підсумувати так:

1.  **Оптимальний вибірковий overhead**: TEM є оптимальним щодо
[теоретичних меж](https://journals.aps.org/prl/abstract/10.1103/PhysRevLett.131.210601),
що означає, що жоден метод не може досягти меншого вибіркового overhead.
Іншими словами, TEM потребує мінімальну кількість додаткових вимірювань для
пом'якшення помилок. Це, у свою чергу, означає, що TEM використовує
мінімальний квантовий час виконання.
2.  **Ефективність витрат**: Оскільки TEM обробляє пом'якшення шуму
повністю на етапі постобробки, немає потреби додавати додаткові схеми до
квантового комп'ютера, що не тільки здешевлює обчислення, але й зменшує
ризик внесення додаткових помилок через недосконалість квантових пристроїв.
3.  **Оцінка кількох спостережуваних величин**: Завдяки інформаційно повним
вимірюванням TEM ефективно оцінює кілька спостережуваних величин з одними
й тими ж даними вимірювань від квантового комп'ютера.
4.  **Пом'якшення помилок вимірювання**: Функція TEM Qiskit також включає
[пропрієтарний метод пом'якшення помилок вимірювання](https://journals.aps.org/prresearch/abstract/10.1103/PhysRevResearch.5.033154),
здатний значно зменшити помилки зчитування після короткого калібрувального
запуску.
5.  **Точність**: TEM значно підвищує точність та надійність цифрових
квантових симуляцій, роблячи квантові алгоритми більш прецизійними та
надійними.
## Опис
Функція TEM дозволяє отримувати очікувані значення з пом'якшенням помилок для
кількох спостережуваних величин на квантовій схемі з мінімальним вибірковим
overhead. Схема вимірюється за допомогою інформаційно повної позитивної
операторно-значної міри (IC-POVM), і зібрані результати вимірювань
обробляються на класичному комп'ютері. Це вимірювання використовується для
виконання методів тензорних мереж та побудови карти інверсії шуму. Функція
застосовує відображення, яке повністю інвертує весь зашумлений ланцюг за
допомогою тензорних мереж для представлення зашумлених шарів.

![TEM schematics](../docs/images/guides/algorithmiq-tem/tem_scheme.svg "Error-mitigated estimation of an observable O via post-processing measurement outcomes of the noisy quantum processor. U and N denote an ideal quantum operation and the associated noise map, which can be generally non-local (and extended to grey boxes). D stands for a tensor of operators that are dual to the effects in the IC measurement. The noise mitigation module M is a tensor network that is efficiently contracted from the middle out. The first iteration of the contraction is represented by the dotted purple line, the second one by the dashed line, and the third one by the solid line.")

Після відправки схем до функції вони транспілюються та оптимізуються для
мінімізації кількості шарів з двокубітними вентилями (найбільш зашумлені
вентилі на квантових пристроях). Шум, що впливає на шари, вивчається через
[Qiskit Runtime](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/noise-learner-noise-learner)
за допомогою розрідженої моделі шуму Паулі-Ліндблада, як описано у E. van den Berg, Z.
Minev, A. Kandala, K. Temme, Nat. Phys. (2023).
[arXiv:2201.09866](https://arxiv.org/abs/2201.09866).

Модель шуму є точним описом шуму на пристрої, здатним уловлювати тонкі
особливості, включаючи перехресні завади між кубітами. Однак шум на
пристроях може коливатися та дрейфувати, і вивчений шум може бути неточним
на момент виконання оцінки. Це може призвести до неточних результатів.
## Початок роботи
Автентифікуйтеся за допомогою свого [ключа API IBM Quantum Platform](http://quantum.cloud.ibm.com/) та виберіть функцію TEM наступним чином. (Цей фрагмент коду передбачає, що ви вже [зберегли свій акаунт](/guides/functions#install-qiskit-functions-catalog-client) у локальному середовищі.)

In [1]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

tem_function_name = "algorithmiq/tem"

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# Load your function
tem = catalog.load(tem_function_name)

## Приклад
Наступний фрагмент коду показує приклад, де TEM використовується для обчислення очікуваних значень спостережуваної величини для простої квантової схеми.

In [2]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp

# Create a quantum circuit
qc = QuantumCircuit(3)
qc.u(0.4, 0.9, -0.3, 0)
qc.u(-0.4, 0.2, 1.3, 1)
qc.u(-1.2, -1.2, 0.3, 2)
for _ in range(2):
    qc.barrier()
    qc.cx(0, 1)
    qc.cx(2, 1)
    qc.barrier()
    qc.u(0.4, 0.9, -0.3, 0)
    qc.u(-0.4, 0.2, 1.3, 1)
    qc.u(-1.2, -1.2, 0.3, 2)

# Define the observables
observable = SparsePauliOp("IYX", 1.0)

# Define the execution options
pub = (qc, [observable])
options = {"default_precision": 0.02}

# Define backend to use. TEM will choose the least-busy device reported by IBM if not specified
backend_name = "ibm_torino"

job = tem.run(pubs=[pub], backend_name=backend_name, options=options)

Використовуйте API Qiskit Serverless для перевірки статусу вашого робочого навантаження Qiskit Function:

In [3]:
print(job.status())

QUEUED


You can return the results as:

In [4]:
result = job.result()
evs = result[0].data.evs

Ви можете отримати результати так:

In [5]:
print(job.result())

PrimitiveResult([PubResult(data=DataBin(evs=np.ndarray(<shape=(1,), dtype=float64>), stds=np.ndarray(<shape=(1,), dtype=float64>)), metadata={'evs_non_mitigated': array([-0.06314623]), 'stds_non_mitigated': array([0.05917147]), 'evs_mitigated_no_readout_mitigation': array([-0.06411205]), 'stds_mitigated_no_readout_mitigation': array([0.05992467]), 'evs_non_mitigated_with_readout_mitigation': array([-0.07028881]), 'stds_non_mitigated_with_readout_mitigation': array([0.06353934])})], metadata={'resource_usage': {'RUNNING: OPTIMIZING_FOR_HARDWARE': {'CPU_TIME': 0.915754}, 'RUNNING: WAITING_FOR_QPU': {'CPU_TIME': 18.804865}, 'RUNNING: POST_PROCESSING': {'CPU_TIME': 10.433445}, 'RUNNING: EXECUTING_QPU': {'QPU_TIME': 159.0}}})


> **Info:** Очікуване значення для безшумної схеми для даного оператора повинно бути приблизно `0.18409094298943401`.
## Вхідні дані
**Параметри**

Name | Type | Description | Required | Default | Example
-- | -- | -- | -- | -- | --
`pubs` | Iterable[EstimatorPubLike] | An iterable of PUB-like (primitive unified bloc) objects, such as tuples `(circuit, observables)` or `(circuit, observables, parameters, precision)`. See [Overview of PUBs](/guides/primitive-input-output#overview-of-pubs) for more information. If a non-ISA circuit is passed, it will be transpiled with optimal settings. If an ISA circuit is passed, it will not be transpiled; in this case, the observable must be defined on the whole QPU. | Yes | N/A | (circuit, observables)
`backend_name` | str | Name of the backend to make the query.| No | If not provided, the least-busy backend will be used. | "ibm_fez"
`options` | dict | Input options. See `Options` section for more details. | No | See `Options` section for more details.| {"max_bond_dimension": 100\}

> **Caution:** TEM наразі має такі обмеження:
> 
>   - Параметризовані схеми не підтримуються. Аргумент parameters повинен бути встановлений у `None`, якщо вказана precision. Це обмеження буде знято у майбутніх версіях.
>   - Підтримуються лише схеми без циклів. Це обмеження буде знято у майбутніх версіях.
>   - Неунітарні вентилі, такі як reset, measure та всі форми потоку керування, не підтримуються. Підтримку reset буде додано у наступних випусках.
### Опції
Словник, що містить розширені опції для TEM. Словник може містити ключі, наведені в таблиці нижче. Якщо будь-яка з опцій не надана, буде використано значення за замовчуванням, вказане в таблиці. Значення за замовчуванням підходять для типового використання TEM.

Name | Choices | Description  | Default
-- | -- | -- | --
`tem_max_bond_dimension` | int | The maximum bond dimension to be used for the tensor networks. | 500 |
`tem_compression_cutoff` | float | The cutoff value to be used for the tensor networks. | 1e-16
`compute_shadows_bias_from_observable` | bool | A boolean flag indicating whether the bias for the classical shadows measurement protocol should be tailored to the PUB observable or not. If False, the classical shadows protocol (equal probability of measuring Z, X, Y) will be used.| False
`shadows_bias` | np.ndarray | The bias to be used for the randomized classical shadows measurement protocol, a 1d or 2d array of size 3 or shape (num_qubits, 3) respectively. order is ZXY | np.array([1 / 3, 1 / 3, 1 / 3])
`max_execution_time` | int or `None` | The maximum execution time on the QPU in seconds. If the runtime exceeds this value, the job will be canceled. If `None`, a default limit set by Qiskit Runtime will apply. | `None`
`num_randomizations` | int | The number of randomizations to be used for noise learning and gate twirling. | 32
`max_layers_to_learn` | int | The maximum number of unique layers to learn. | 4
`mitigate_readout_error` | bool | A Boolean flag indicating whether to perform readout error mitigation or not. | True
`num_readout_calibration_shots` | int | The number of shots to be used for readout error mitigation. | 10000
`default_precision` | float | The default precision to be used for the PUBs for which the precision is not specified. |0.02
`seed` | int or `None` | Set the seed of the random number generator for reproducibility. If `None`, don't set the seed. | None
## Вихідні дані
Qiskit [PrimitiveResults](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PrimitiveResult), що містить результат з пом'якшенням TEM. Результат для кожного PUB повертається як [PubResult](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PubResult), що містить наступні поля:

Name |Type | Description
-- | -- | --
data | DataBin | A Qiskit [DataBin](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.DataBin) containing the TEM mitigated observable and its standard error. The DataBin has the following fields: <ul><li>`evs`: The TEM-mitigated observable value.</li><li>`stds`: The standard error of the TEM-mitigated observable.</li></ul>
metadata | dict | A dictionary containing additional results. The dictionary contains the following keys: <ul><li>`"evs_non_mitigated"`: The observable value without error mitigation.</li><li>`"stds_non_mitigated"`: The standard error of the result without error mitigation.</li><li>`"evs_mitigated_no_readout_mitigation"`: The observable value with error mitigation but without readout error mitigation.</li><li>`"stds_mitigated_no_readout_mitigation"`: The standard error of the result with error mitigation but without readout error mitigation.</li><li>`"evs_non_mitigated_with_readout_mitigation"`: The observable value without error mitigation but with readout error mitigation.</li><li>`"stds_non_mitigated_with_readout_mitigation"`: The standard error of the result without error mitigation but with readout error mitigation.</li></ul>
## Отримання повідомлень про помилки
Якщо статус вашого робочого навантаження — ERROR, використовуйте job.result() для отримання повідомлення про помилку наступним чином: